In [ ]:
import sys
from pathlib import Path

# Find project root dynamically
def get_project_root() -> Path:
    try:
        path = Path(__file__).resolve()
        for parent in [path] + list(path.parents):
            if (parent / "requirements.txt").exists() or (parent / "project").exists():
                return parent
    except NameError:
        pass
    path = Path.cwd().resolve()
    for parent in [path] + list(path.parents):
        if (parent / "requirements.txt").exists() or (parent / "project").exists():
            return parent
    return path

ROOT = get_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import torch.nn as nn
import torch.nn.functional as F


class InfoNCELoss(nn.Module):
    def __init__(self, init_temperature=0.07):
        """
        Args:
            init_temperature: Initial value of tau.
                              CLIP initializes tau around 0.07.
        """
        super().__init__()

        # Learn log(1/tau) instead of tau itself.
        # This guarantees tau always remains positive after exponentiation.
        self.log_inv_tau = nn.Parameter(
            torch.tensor([1.0 / init_temperature]).log()
        )

    def forward(self, image_embeds, text_embeds):
        """
        Args:
            image_embeds : (B, D)
            text_embeds  : (B, D)

        Returns:
            Symmetric InfoNCE loss.
        """

        # Step 1: L2-normalize embeddings
        # Makes dot product equal cosine similarity.
        image_embeds = F.normalize(image_embeds, dim=-1)
        text_embeds = F.normalize(text_embeds, dim=-1)

        # Step 2: Compute inverse temperature
        # Clamp log_inv_tau for numerical stability.
        # log(100) ≈ 4.6052
        log_inv_tau = self.log_inv_tau.clamp(0, 4.6052)
        inv_tau = log_inv_tau.exp()

        # Step 3: Similarity matrix
        # image_embeds : (B, D)
        # text_embeds.T: (D, B)
        # logits       : (B, B)
        # logits[i][j] =
        # cosine_similarity(image_i, text_j) / tau
        logits = inv_tau * (image_embeds @ text_embeds.T)

        # Step 4: Ground-truth labels
        # Correct pairs lie along the diagonal.
        # labels = [0,1,2,...,B-1]
        batch_size = image_embeds.size(0)
        labels = torch.arange(batch_size, device=logits.device)

        # Step 5: Image -> Text loss
        # Each image should retrieve its matching caption.
        loss_i2t = F.cross_entropy(logits, labels)

        # Step 6: Text -> Image loss
        # Each caption should retrieve its matching image.
        loss_t2i = F.cross_entropy(logits.T, labels)

        # Step 7: Symmetric CLIP loss
        loss = (loss_i2t + loss_t2i) / 2

        return loss